# Account-level SNA Feature Engineering (MVP 12)

根據 `strategy/account_SNA_feature.md` 實作帳戶級 SNA MVP（12 特徵）。

- 任務仍是交易級預測 `isFraud`
- 特徵來自來源/目的帳戶在當下之前的歷史狀態
- 嚴格 leakage-safe：每筆先取特徵再更新統計


In [ ]:
# ==== 0) Imports ====
import warnings
warnings.filterwarnings('ignore')

from collections import defaultdict
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

import matplotlib.pyplot as plt

# Optional boosters
HAS_LGB = True
HAS_XGB = True
try:
    import lightgbm as lgb
except Exception:
    HAS_LGB = False

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGB = False


In [ ]:
# ==== 1) Config ====
DATA_PATH = '../Transactions Data.csv'

USE_SAMPLE = True
SAMPLE_FRAC = 0.20
RANDOM_STATE = 42
MAX_ROWS = None  # e.g. 500_000

TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO  = 0.15


In [ ]:
# ==== 2) Load data ====
df = pd.read_csv(DATA_PATH)
if MAX_ROWS is not None:
    df = df.iloc[:MAX_ROWS].copy()

if USE_SAMPLE:
    sampled = []
    for _, g in df.groupby('step', sort=False):
        if len(g) <= 1:
            sampled.append(g)
        else:
            sampled.append(g.sample(frac=min(SAMPLE_FRAC, 1.0), random_state=RANDOM_STATE))
    df = pd.concat(sampled, ignore_index=True)

df = df.sort_values('step').reset_index(drop=True)

print('Shape:', df.shape)
print('Fraud rate:', df['isFraud'].mean())
print(df.head(3))


## 3) Build Account-level SNA MVP 12 Features (Leakage-safe)

12 features:
- orig_out_degree_hist
- orig_unique_dest_hist
- dest_in_degree_hist
- dest_unique_orig_hist
- orig_out_amount_sum_hist
- orig_out_amount_mean_hist
- dest_in_amount_sum_hist
- dest_in_amount_mean_hist
- orig_repeat_counterparty_ratio_hist
- dest_repeat_counterparty_ratio_hist
- orig_activity_burst_proxy_hist
- dest_activity_burst_proxy_hist


In [ ]:
# ==== 3) Incremental account-SNA feature generation ====
work = df.copy()

# account states
orig_out_degree = defaultdict(int)
dest_in_degree = defaultdict(int)
orig_out_amount_sum = defaultdict(float)
dest_in_amount_sum = defaultdict(float)
orig_to_dests = defaultdict(set)
dest_from_origs = defaultdict(set)

n = len(work)

# allocate arrays
f_orig_out_degree = np.zeros(n, dtype=np.int32)
f_orig_unique_dest = np.zeros(n, dtype=np.int32)
f_dest_in_degree = np.zeros(n, dtype=np.int32)
f_dest_unique_orig = np.zeros(n, dtype=np.int32)

f_orig_out_amount_sum = np.zeros(n, dtype=np.float64)
f_orig_out_amount_mean = np.zeros(n, dtype=np.float64)
f_dest_in_amount_sum = np.zeros(n, dtype=np.float64)
f_dest_in_amount_mean = np.zeros(n, dtype=np.float64)

f_orig_repeat_ratio = np.zeros(n, dtype=np.float64)
f_dest_repeat_ratio = np.zeros(n, dtype=np.float64)
f_orig_burst_proxy = np.zeros(n, dtype=np.float64)
f_dest_burst_proxy = np.zeros(n, dtype=np.float64)

orig_col = work['nameOrig'].values
dest_col = work['nameDest'].values
amt_col = work['amount'].values
step_col = work['step'].values

for i in range(n):
    o = orig_col[i]
    d = dest_col[i]
    a = float(amt_col[i])
    s = max(int(step_col[i]), 1)

    # ----- read history (t_j < t_i) -----
    o_deg = orig_out_degree[o]
    d_deg = dest_in_degree[d]

    o_uniq = len(orig_to_dests[o])
    d_uniq = len(dest_from_origs[d])

    o_sum = orig_out_amount_sum[o]
    d_sum = dest_in_amount_sum[d]

    f_orig_out_degree[i] = o_deg
    f_orig_unique_dest[i] = o_uniq
    f_dest_in_degree[i] = d_deg
    f_dest_unique_orig[i] = d_uniq

    f_orig_out_amount_sum[i] = o_sum
    f_dest_in_amount_sum[i] = d_sum

    f_orig_out_amount_mean[i] = o_sum / max(o_deg, 1)
    f_dest_in_amount_mean[i] = d_sum / max(d_deg, 1)

    f_orig_repeat_ratio[i] = 1.0 - (o_uniq / max(o_deg, 1))
    f_dest_repeat_ratio[i] = 1.0 - (d_uniq / max(d_deg, 1))

    f_orig_burst_proxy[i] = o_deg / s
    f_dest_burst_proxy[i] = d_deg / s

    # ----- update states with current txn -----
    orig_out_degree[o] += 1
    dest_in_degree[d] += 1

    orig_out_amount_sum[o] += a
    dest_in_amount_sum[d] += a

    orig_to_dests[o].add(d)
    dest_from_origs[d].add(o)

# assign back
work['orig_out_degree_hist'] = f_orig_out_degree
work['orig_unique_dest_hist'] = f_orig_unique_dest
work['dest_in_degree_hist'] = f_dest_in_degree
work['dest_unique_orig_hist'] = f_dest_unique_orig

work['orig_out_amount_sum_hist'] = f_orig_out_amount_sum
work['orig_out_amount_mean_hist'] = f_orig_out_amount_mean
work['dest_in_amount_sum_hist'] = f_dest_in_amount_sum
work['dest_in_amount_mean_hist'] = f_dest_in_amount_mean

work['orig_repeat_counterparty_ratio_hist'] = f_orig_repeat_ratio
work['dest_repeat_counterparty_ratio_hist'] = f_dest_repeat_ratio
work['orig_activity_burst_proxy_hist'] = f_orig_burst_proxy
work['dest_activity_burst_proxy_hist'] = f_dest_burst_proxy

print('Added 12 account-SNA features.')
work[['orig_out_degree_hist','dest_in_degree_hist','orig_repeat_counterparty_ratio_hist']].head()


In [ ]:
# ==== 4) Add base engineered features ====
work['deltaOrig'] = work['oldbalanceOrg'] - work['newbalanceOrig']
work['deltaDest'] = work['newbalanceDest'] - work['oldbalanceDest']
work['isOrigBalanceZero'] = (work['oldbalanceOrg'] == 0).astype(int)
work['isDestBalanceZero'] = (work['oldbalanceDest'] == 0).astype(int)

TARGET = 'isFraud'


In [ ]:
# ==== 5) Time-based split ====
steps_sorted = np.sort(work['step'].unique())
train_cut_idx = int(len(steps_sorted) * TRAIN_RATIO)
valid_cut_idx = int(len(steps_sorted) * (TRAIN_RATIO + VALID_RATIO))

train_max_step = steps_sorted[train_cut_idx - 1]
valid_max_step = steps_sorted[valid_cut_idx - 1]

train_mask = work['step'] <= train_max_step
valid_mask = (work['step'] > train_max_step) & (work['step'] <= valid_max_step)
test_mask = work['step'] > valid_max_step

print('Train:', train_mask.sum(), 'fraud rate:', work.loc[train_mask, TARGET].mean())
print('Valid:', valid_mask.sum(), 'fraud rate:', work.loc[valid_mask, TARGET].mean())
print('Test :', test_mask.sum(), 'fraud rate:', work.loc[test_mask, TARGET].mean())


In [ ]:
# ==== 6) Feature sets: Base vs Base+AccountSNA ====
base_num = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud',
    'deltaOrig', 'deltaDest', 'isOrigBalanceZero', 'isDestBalanceZero'
]
base_cat = ['type']

acc_sna_num = [
    'orig_out_degree_hist', 'orig_unique_dest_hist',
    'dest_in_degree_hist', 'dest_unique_orig_hist',
    'orig_out_amount_sum_hist', 'orig_out_amount_mean_hist',
    'dest_in_amount_sum_hist', 'dest_in_amount_mean_hist',
    'orig_repeat_counterparty_ratio_hist', 'dest_repeat_counterparty_ratio_hist',
    'orig_activity_burst_proxy_hist', 'dest_activity_burst_proxy_hist'
]

base_cols = base_num + base_cat
base_acc_cols = base_num + base_cat + acc_sna_num

X_base = work[base_cols].copy()
X_base_acc = work[base_acc_cols].copy()
y = work[TARGET].values

y_train = y[train_mask]
y_valid = y[valid_mask]
y_test = y[test_mask]


In [ ]:
# ==== 7) Eval helpers ====
def evaluate_probs(y_true, y_prob, name='model'):
    ap = average_precision_score(y_true, y_prob)
    roc = roc_auc_score(y_true, y_prob)
    print(f'[{name}] PR-AUC={ap:.6f} | ROC-AUC={roc:.6f}')
    return {'model': name, 'pr_auc': ap, 'roc_auc': roc}

def plot_pr(y_true, y_prob, title='PR curve'):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    plt.figure(figsize=(6,4))
    plt.plot(r, p, label=f'AP={ap:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.grid(alpha=0.2)
    plt.legend()
    plt.show()


In [ ]:
# ==== 8) Logistic: Base vs Base+AccountSNA ====
pre_base = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), base_num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), base_cat)
])

pre_base_acc = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), base_num + acc_sna_num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), base_cat)
])

logit_base = Pipeline([
    ('prep', pre_base),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='saga', n_jobs=-1, random_state=RANDOM_STATE))
])

logit_base_acc = Pipeline([
    ('prep', pre_base_acc),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='saga', n_jobs=-1, random_state=RANDOM_STATE))
])

Xb_train, Xb_valid = X_base.loc[train_mask], X_base.loc[valid_mask]
Xba_train, Xba_valid = X_base_acc.loc[train_mask], X_base_acc.loc[valid_mask]

logit_base.fit(Xb_train, y_train)
logit_base_acc.fit(Xba_train, y_train)

p_base = logit_base.predict_proba(Xb_valid)[:,1]
p_base_acc = logit_base_acc.predict_proba(Xba_valid)[:,1]

rows=[]
rows.append(evaluate_probs(y_valid, p_base, 'Logit-Base (Valid)'))
rows.append(evaluate_probs(y_valid, p_base_acc, 'Logit-Base+AccountSNA (Valid)'))

plot_pr(y_valid, p_base, 'Logit Base - Valid PR')
plot_pr(y_valid, p_base_acc, 'Logit Base+AccountSNA - Valid PR')

pd.DataFrame(rows)


In [ ]:
# ==== 9) LightGBM (optional): Base+AccountSNA ====
if HAS_LGB:
    lgb_features = base_num + acc_sna_num + ['type']
    lgb_df = work[lgb_features + [TARGET]].copy()
    lgb_df['type'] = lgb_df['type'].astype('category')

    train_df = lgb_df.loc[train_mask]
    valid_df = lgb_df.loc[valid_mask]

    X_train_lgb = train_df.drop(columns=[TARGET])
    y_train_lgb = train_df[TARGET]
    X_valid_lgb = valid_df.drop(columns=[TARGET])
    y_valid_lgb = valid_df[TARGET]

    neg = (y_train_lgb == 0).sum()
    pos = (y_train_lgb == 1).sum()
    spw = max(1.0, neg / max(pos, 1))

    model_lgb = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=64,
        min_child_samples=100,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model_lgb.fit(
        X_train_lgb, y_train_lgb,
        eval_set=[(X_valid_lgb, y_valid_lgb)],
        eval_metric='aucpr',
        categorical_feature=['type'],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    p_lgb = model_lgb.predict_proba(X_valid_lgb)[:, 1]
    _ = evaluate_probs(y_valid_lgb, p_lgb, 'LightGBM-Base+AccountSNA (Valid)')
    plot_pr(y_valid_lgb, p_lgb, 'LightGBM Base+AccountSNA - Valid PR')
else:
    print('LightGBM not installed; skip this block.')


In [ ]:
# ==== 10) XGBoost (optional): Base+AccountSNA ====
if HAS_XGB:
    xgb_df = work[base_num + base_cat + acc_sna_num + [TARGET]].copy()
    xgb_df = pd.get_dummies(xgb_df, columns=['type'], drop_first=False)

    train_df = xgb_df.loc[train_mask]
    valid_df = xgb_df.loc[valid_mask]

    X_train_xgb = train_df.drop(columns=[TARGET])
    y_train_xgb = train_df[TARGET]
    X_valid_xgb = valid_df.drop(columns=[TARGET])
    y_valid_xgb = valid_df[TARGET]

    neg = (y_train_xgb == 0).sum()
    pos = (y_train_xgb == 1).sum()
    spw = max(1.0, neg / max(pos, 1))

    model_xgb = XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method='hist'
    )

    model_xgb.fit(X_train_xgb, y_train_xgb, eval_set=[(X_valid_xgb, y_valid_xgb)], verbose=False)

    p_xgb = model_xgb.predict_proba(X_valid_xgb)[:,1]
    _ = evaluate_probs(y_valid_xgb, p_xgb, 'XGBoost-Base+AccountSNA (Valid)')
    plot_pr(y_valid_xgb, p_xgb, 'XGBoost Base+AccountSNA - Valid PR')
else:
    print('XGBoost not installed; skip this block.')


## 下一步

1. 先看 Logit Base vs Base+AccountSNA 的 PR-AUC 差異。  
2. 若有增益，再用 LightGBM/XGBoost 做 threshold 與風險分級。  
3. 若資源不足，先保留增益最大的 account-SNA 特徵。  
